In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
import gc
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import lightautoml

from lightautoml.automl.presets.tabular_presets import TabularAutoML, TabularUtilizedAutoML
from lightautoml.tasks import Task

In [3]:
print(lightautoml.__version__)

0.4.2


In [4]:
pf = pq.ParquetFile('data_train_std.pq')
pf.num_row_groups

3

In [5]:
for i in range(pf.num_row_groups):
    num_rows_in_rg = pf.metadata.row_group(i).num_rows
    print(f'Row group {i} содержит {num_rows_in_rg} строк')

Row group 0 содержит 1048576 строк
Row group 1 содержит 1048576 строк
Row group 2 содержит 302848 строк


In [6]:
table = pf.read_row_group(0)
table.shape

(1048576, 388)

In [7]:
df_batch = table.to_pandas()
df_batch.shape

(1048576, 388)

In [10]:
df_batch = df_batch.drop(['id'], axis=1)
df_batch.shape

(1048576, 387)

In [11]:
task = Task('binary')

In [14]:
timeout = 3600 * 3
threads = 4
cv = 5
rs = 42
target = 'flag'

In [15]:
roles = {
    'target': target
}

In [16]:
automl = TabularAutoML(task = task,
                       timeout = timeout,
                       cpu_limit = threads,
                       reader_params = {'n_jobs': threads, 'random_state': rs, 'cv': cv})

In [17]:
%%time
oof_pred = automl.fit_predict(df_batch, roles = roles, verbose = 3)

[12:22:52] Stdout logging level is INFO3.
[12:22:52] Copying TaskTimer may affect the parent PipelineTimer, so copy will create new unlimited TaskTimer
[12:22:52] Task: binary

[12:22:52] Start automl preset with listed constraints:
[12:22:52] - time: 10800.00 seconds
[12:22:52] - CPU: 4 cores
[12:22:52] - memory: 16 GB

[12:22:52] Train data shape: (1048576, 387)

[12:23:19] Feats was rejected during automatic roles guess: []
[12:23:20] Layer 1 train process start. Time left 10771.67 secs
[12:24:51] Start fitting Lvl_0_Pipe_0_Mod_0_LinearL2 ...
[12:25:41] ===== Start working with fold 0 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[12:25:50] Linear model: C = 1e-05 score = 0.7208434574496715
[12:25:53] Linear model: C = 5e-05 score = 0.726298643099768
[12:25:56] Linear model: C = 0.0001 score = 0.7274517926482785
[12:26:02] Linear model: C = 0.0005 score = 0.7293556278793564
[12:26:03] Linear model: C = 0.001 score = 0.7293556278793564
[12:26:03] Linear model: C = 0.005 score = 0.72935562787

In [23]:
automl.levels

[[<lightautoml.pipelines.ml.nested_ml_pipe.NestedTabularMLPipeline at 0x72e5803ffa90>,